# 03 - ICCU integration

Controllo delle relazioni tra i dataset ICCU secondari e del parsing delle confluenze bibliotecarie.

### Riproducibilità

Questo notebook documenta e verifica la fase di integrazione dei dataset ICCU.

Le trasformazioni sono implementate negli script della directory `scripts/` e utilizzano i dati RAW originali conservati nel repository.

Il notebook permette di verificare le relazioni tra biblioteche, collezioni, patrimoni e confluenze mantenendo gli identificatori ICCU come chiavi principali.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    for c in candidates:
        if (c / "data" / "processed").exists() and (c / "metadata").exists():
            return c
    raise FileNotFoundError("Eseguire il notebook dalla root del repository o da notebooks/.")

ROOT = project_root()
ROOT

## Integrazione delle relazioni ICCU 1:N
Le relazioni non vengono appiattite: patrimonio, fondi, contatti e denominazioni precedenti mantengono la granularità originale.

In [2]:
files = {
    "types": "library_type.csv",
    "holdings": "library_holdings.csv",
    "funds": "special_collection.csv",
    "contacts": "library_contact.csv",
    "previous_names": "library_previous_name.csv",
    "mergers": "library_mergers.csv",
}
tables = {k: pd.read_csv(ROOT / "data/processed" / v, dtype=str, keep_default_na=False, low_memory=False) for k,v in files.items()}
pd.DataFrame({"resource": list(tables), "rows": [len(tables[k]) for k in tables]})

,resource,rows
0,types,13715
1,holdings,93512
2,funds,9737
3,contacts,62804
4,previous_names,9507
5,mergers,1502


In [3]:
assert len(tables["types"]) == 13715
assert len(tables["holdings"]) == 93512
assert len(tables["funds"]) == 9737
assert len(tables["contacts"]) == 62804
assert len(tables["previous_names"]) == 9507
assert len(tables["mergers"]) == 1502
print("OK: cardinalità 1:N canoniche verificate.")

OK: cardinalità 1:N canoniche verificate.


In [4]:
m = tables["mergers"]
summary = {
    "merged_records": len(m),
    "parse_success": int((m["parse_success"] == "True").sum()),
    "target_exists": int((m["target_exists_in_snapshot"] == "True").sum()),
    "self_loops": int((m["self_loop"] == "True").sum()),
    "cycles": int((m["in_cycle"] == "True").sum()),
}
assert summary == {"merged_records":1502,"parse_success":1412,"target_exists":1412,"self_loops":1,"cycles":1}
summary

{'merged_records': 1502,
 'parse_success': 1412,
 'target_exists': 1412,
 'self_loops': 1,
 'cycles': 1}